In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx
# from shapely.geometry import Point, Polygon
from shapely.geometry import box
import rasterio
# import rasterio.plot
from rasterio.warp import transform_bounds, reproject, Resampling
from rasterio.windows import from_bounds
from matplotlib.patches import Patch


pd.options.display.max_colwidth = 100
pd.options.display.max_rows = 10
pd.options.display.max_columns = 30

In [ ]:
# read in data

# evac data: dissolve to one multipolygon because we dont care about warning/order differentiating
evac_raw = gpd.read_parquet('../01_data/02_clean/evac_clean_aggregated.parquet')
evac_raw = evac_raw[['geometry']]
evac_raw = evac_raw.dissolve()
evac_crs = evac_raw.crs

# fires
fires = gpd.read_file('../01_data/01_raw/data_2025_01_17.geojson').to_crs(epsg=2229)
fires["poly_DateCurrent"] = fires["poly_DateCurrent"].dt.tz_convert('US/Pacific')
fires = fires[fires['poly_DateCurrent'] > '2025-01-06']
fires["poly_DateCurrent"] = fires["poly_DateCurrent"].dt.date
fires = fires[['geometry']]
fires_union = fires.dissolve()  # dissolve to one multipolygon
fires_union = fires_union.to_crs(evac_crs)  # convert to same CRS as evac data

# CA state boundary for trimming coastal CT
states = gpd.read_file('../01_data/01_raw/cb_2018_us_state_500k.shp')
ca_state = states[states['STUSPS'] == 'CA'].reset_index(drop=True)
ca_state = ca_state[['geometry']]
ca_state = ca_state.to_crs(evac_crs)

# census tracts: just need to send data to kpsc with ct geoid
cts = gpd.read_file('../01_data/01_raw/tl_2010_06_tract10.shp')
cts = cts[['GEOID10', 'geometry']].rename(columns={'GEOID10': 'geoid'})
cts = cts.to_crs(evac_crs)
cts = gpd.overlay(cts, ca_state, how='intersection') # intersect with CA state boundary

# pop data path (only going to read in as needed, so doing so below)
ghs_path = '../01_data/01_raw/GHS_POP_E2025_GLOBE_R2023A_54009_100_V1_0/GHS_POP_E2025_GLOBE_R2023A_54009_100_V1_0.tif'

In [ ]:
# determine exposed CTs
# using overlap so that its not just CTs that touch the evacuation area, but also those that are within it
# need the extra `within` because some CTs are completely within the evacuation area

exposed_cts = cts[cts.overlaps(evac_raw.unary_union) | cts.within(evac_raw.unary_union)]

In [ ]:
# map! 

# convert all geometries to Web Mercator for plotting
cts_mercator = cts.to_crs('EPSG:3857')
exposed_cts_mercator = exposed_cts.to_crs('EPSG:3857')
evac_mercator = evac_raw.to_crs('EPSG:3857')
fires_mercator = fires_union.to_crs('EPSG:3857')

fig, ax = plt.subplots(1, 1, figsize=(15, 12))

ax_crs = cts.crs
# convert to Web Mercator for contextily
cts_mercator = cts.to_crs('EPSG:3857')
exposed_cts_mercator = exposed_cts.to_crs('EPSG:3857')
evac_mercator = evac_raw.to_crs('EPSG:3857')
fires_mercator = fires_union.to_crs('EPSG:3857')

# create bounds based on exposed census tracts area
combined_bounds_geom = exposed_cts_mercator.union_all()

# get bounds for clipping and add some buffer
minx, miny, maxx, maxy = combined_bounds_geom.bounds
buffer_x = (maxx - minx) * 0.1  # 10% buffer
buffer_y = (maxy - miny) * 0.1
clip_bounds = [minx - buffer_x, miny - buffer_y, maxx + buffer_x, maxy + buffer_y]

# clip all census tracts to the area of interest
clip_box = box(*clip_bounds)
cts_clipped = cts_mercator[cts_mercator.intersects(clip_box)]

# plot all census tracts (outline only, no fill)
cts_clipped.plot(ax=ax, facecolor='none', edgecolor='gray', linewidth=0.5, alpha=0.7)

# plot intersecting census tracts in green
exposed_cts_mercator.plot(ax=ax, color='#0C5985', edgecolor='none', linewidth=1)

# plot evacuation zones
evac_mercator.plot(ax=ax, edgecolor='#66A8CF', color='none', linewidth=2)

# plot fire boundaries (red outlines only)
fires_mercator.boundary.plot(ax=ax, color='#8f1402', linewidth=2, alpha=0.8)

# set the axis limits to the clipped bounds
ax.set_xlim(clip_bounds[0], clip_bounds[2])
ax.set_ylim(clip_bounds[1], clip_bounds[3])
    
# ctx.add_basemap(ax, crs='EPSG:3857', source=ctx.providers.OpenStreetMap.Mapnik, alpha=0.6)

legend_elements = [
    Patch(facecolor='none', edgecolor='gray', linewidth=0.5, label='Census tracts'),
    Patch(facecolor='#0C5985', label='Evacuation exposed census tracts'),
    Patch(facecolor='none', edgecolor='#66A8CF', linewidth=2, label='Evacuation zone'),
    Patch(facecolor='none', edgecolor='#8f1402', linewidth=2, label='Fire boundaries')
]
ax.legend(handles=legend_elements, loc='upper right', fontsize=16)

ax.set_xticks([])
ax.set_yticks([])

plt.tight_layout()
plt.show()


In [ ]:
exposed_cts['exposed_evac'] = 1

# save as parquet and csv
exposed_cts.to_parquet('../01_data/02_clean/exposed_cts_evac.parquet', index=False)
exposed_cts[['geoid', 'exposed_evac']].to_csv('../01_data/02_clean/exposed_cts_evac.csv', index=False)

In [ ]:
# map with pop to check exposed CTs

# convert all geometries to Web Mercator for plotting
cts_mercator = cts.to_crs('EPSG:3857')
exposed_cts_mercator = exposed_cts.to_crs('EPSG:3857')
evac_mercator = evac_raw.to_crs('EPSG:3857')
fires_mercator = fires_union.to_crs('EPSG:3857')

fig, ax = plt.subplots(1, 1, figsize=(15, 12))

ax_crs = cts.crs
# convert to Web Mercator for contextily
cts_mercator = cts.to_crs('EPSG:3857')
exposed_cts_mercator = exposed_cts.to_crs('EPSG:3857')
evac_mercator = evac_raw.to_crs('EPSG:3857')
fires_mercator = fires_union.to_crs('EPSG:3857')

# create bounds based on exposed census tracts area
combined_bounds_geom = exposed_cts_mercator.union_all()

# get bounds for clipping and add some buffer
minx, miny, maxx, maxy = combined_bounds_geom.bounds
buffer_x = (maxx - minx) * 0.1  # 10% buffer
buffer_y = (maxy - miny) * 0.1
clip_bounds = [minx - buffer_x, miny - buffer_y, maxx + buffer_x, maxy + buffer_y]

# Read and process the GHS raster data
with rasterio.open(ghs_path) as src:
    # Transform clip bounds to the raster's CRS
    raster_bounds = transform_bounds('EPSG:3857', src.crs, *clip_bounds)
    
    # Create a window for the area of interest
    window = from_bounds(*raster_bounds, src.transform)
    
    # Read the data for this window
    pop_data = src.read(1, window=window)
    
    # Get the transform for this window
    window_transform = src.window_transform(window)
    
    # Calculate destination bounds and shape in Web Mercator
    dst_width = int((clip_bounds[2] - clip_bounds[0]) / 100)  # 100m resolution
    dst_height = int((clip_bounds[3] - clip_bounds[1]) / 100)
    
    # Create destination array
    dst_array = np.zeros((dst_height, dst_width), dtype=np.float32)
    
    # Create destination transform
    dst_transform = rasterio.transform.from_bounds(
        clip_bounds[0], clip_bounds[1], clip_bounds[2], clip_bounds[3],
        dst_width, dst_height
    )
    
    # Reproject to Web Mercator
    reproject(
        source=pop_data,
        destination=dst_array,
        src_transform=window_transform,
        src_crs=src.crs,
        dst_transform=dst_transform,
        dst_crs='EPSG:3857',
        resampling=Resampling.bilinear
    )

# Mask zero and negative values
dst_array = np.where(dst_array <= 0, np.nan, dst_array)

# Plot population data as background
pop_extent = [clip_bounds[0], clip_bounds[2], clip_bounds[1], clip_bounds[3]]
im = ax.imshow(dst_array, extent=pop_extent, cmap='YlOrRd', alpha=0.7, 
               vmin=0, vmax=np.nanpercentile(dst_array, 95))

# clip all census tracts to the area of interest
clip_box = box(*clip_bounds)
cts_clipped = cts_mercator[cts_mercator.intersects(clip_box)]

# plot all census tracts (outline only, no fill)
cts_clipped.plot(ax=ax, facecolor='none', edgecolor='gray', linewidth=0.5, alpha=0.7)

# plot intersecting census tracts in green
exposed_cts_mercator.plot(ax=ax, color='#4a6741', alpha=0.7, edgecolor='#4a6741', linewidth=1)

# plot evacuation zones (filled, no outline)
evac_mercator.plot(ax=ax, color='#29505d', alpha=0.5, edgecolor='none')

# plot fire boundaries (red outlines only)
fires_mercator.boundary.plot(ax=ax, color='#8f1402', linewidth=2, alpha=0.8)

# set the axis limits to the clipped bounds
ax.set_xlim(clip_bounds[0], clip_bounds[2])
ax.set_ylim(clip_bounds[1], clip_bounds[3])
    
ctx.add_basemap(ax, crs='EPSG:3857', source=ctx.providers.OpenStreetMap.Mapnik, alpha=0.3)

# Add colorbar for population
cbar = plt.colorbar(im, ax=ax, shrink=0.6, aspect=20)
cbar.set_label('Population Density (people per 100m cell)', fontsize=10)

ax.set_title('Census tract-level evacuation boundary exposure, with fire boundaries', fontsize=16, fontweight='bold')

legend_elements = [
    Patch(facecolor='none', edgecolor='gray', linewidth=0.5, label='Census Tracts'),
    Patch(facecolor='#4a6741', alpha=0.5, label='Exposed Census Tracts'),
    Patch(facecolor='#29505d', alpha=0.5, label='Evacuation Zone'),
    Patch(facecolor='none', edgecolor='#8f1402', linewidth=2, label='Fire Boundaries'),
]
ax.legend(handles=legend_elements, loc='upper right')

ax.set_xticks([])
ax.set_yticks([])

plt.tight_layout()
plt.show()

In [ ]:
# map with pop and just outlines for better visuals

# convert all geometries to Web Mercator for plotting
cts_mercator = cts.to_crs('EPSG:3857')
exposed_cts_mercator = exposed_cts.to_crs('EPSG:3857')
evac_mercator = evac_raw.to_crs('EPSG:3857')
fires_mercator = fires_union.to_crs('EPSG:3857')

fig, ax = plt.subplots(1, 1, figsize=(15, 12))

ax_crs = cts.crs
# convert to Web Mercator for contextily
cts_mercator = cts.to_crs('EPSG:3857')
exposed_cts_mercator = exposed_cts.to_crs('EPSG:3857')
evac_mercator = evac_raw.to_crs('EPSG:3857')
fires_mercator = fires_union.to_crs('EPSG:3857')

# create bounds based on exposed census tracts area
combined_bounds_geom = exposed_cts_mercator.union_all()

# get bounds for clipping and add some buffer
minx, miny, maxx, maxy = combined_bounds_geom.bounds
buffer_x = (maxx - minx) * 0.1  # 10% buffer
buffer_y = (maxy - miny) * 0.1
clip_bounds = [minx - buffer_x, miny - buffer_y, maxx + buffer_x, maxy + buffer_y]

# Read and process the GHS raster data
with rasterio.open(ghs_path) as src:
    # Transform clip bounds to the raster's CRS
    raster_bounds = transform_bounds('EPSG:3857', src.crs, *clip_bounds)
    
    # Create a window for the area of interest
    window = from_bounds(*raster_bounds, src.transform)
    
    # Read the data for this window
    pop_data = src.read(1, window=window)
    
    # Get the transform for this window
    window_transform = src.window_transform(window)
    
    # Calculate destination bounds and shape in Web Mercator
    dst_width = int((clip_bounds[2] - clip_bounds[0]) / 100)  # 100m resolution
    dst_height = int((clip_bounds[3] - clip_bounds[1]) / 100)
    
    # Create destination array
    dst_array = np.zeros((dst_height, dst_width), dtype=np.float32)
    
    # Create destination transform
    dst_transform = rasterio.transform.from_bounds(
        clip_bounds[0], clip_bounds[1], clip_bounds[2], clip_bounds[3],
        dst_width, dst_height
    )
    
    # Reproject to Web Mercator
    reproject(
        source=pop_data,
        destination=dst_array,
        src_transform=window_transform,
        src_crs=src.crs,
        dst_transform=dst_transform,
        dst_crs='EPSG:3857',
        resampling=Resampling.bilinear
    )

# Mask zero and negative values
dst_array = np.where(dst_array <= 0, np.nan, dst_array)

# Plot population data as background
pop_extent = [clip_bounds[0], clip_bounds[2], clip_bounds[1], clip_bounds[3]]
im = ax.imshow(dst_array, extent=pop_extent, cmap='YlOrRd', alpha=0.7, 
               vmin=0, vmax=np.nanpercentile(dst_array, 95))

# clip all census tracts to the area of interest
clip_box = box(*clip_bounds)
cts_clipped = cts_mercator[cts_mercator.intersects(clip_box)]

# plot all census tracts (outline only, no fill)
cts_clipped.plot(ax=ax, facecolor='none', edgecolor='gray', linewidth=0.5, alpha=0.7)

# plot intersecting census tracts in green (outline only)
exposed_cts_mercator.plot(ax=ax, facecolor='none', edgecolor='#4a6741', linewidth=2, alpha=0.9)

# plot evacuation zones (outline only)
evac_mercator.boundary.plot(ax=ax, color='#29505d', linewidth=2, alpha=0.8)

# plot fire boundaries (red outlines only)
fires_mercator.boundary.plot(ax=ax, color='#8f1402', linewidth=2, alpha=0.8)

# set the axis limits to the clipped bounds
ax.set_xlim(clip_bounds[0], clip_bounds[2])
ax.set_ylim(clip_bounds[1], clip_bounds[3])
    
ctx.add_basemap(ax, crs='EPSG:3857', source=ctx.providers.OpenStreetMap.Mapnik, alpha=0.3)

# Add colorbar for population
cbar = plt.colorbar(im, ax=ax, shrink=0.6, aspect=20)
cbar.set_label('Population Density (people per 100m cell)', fontsize=10)

ax.set_title('Census tract-level evacuation boundary exposure, with fire boundaries', fontsize=16, fontweight='bold')

legend_elements = [
    Patch(facecolor='none', edgecolor='gray', linewidth=0.5, label='Census Tracts'),
    Patch(facecolor='none', edgecolor='#4a6741', linewidth=2, label='Exposed Census Tracts'),
    Patch(facecolor='none', edgecolor='#29505d', linewidth=2, label='Evacuation Zone'),
    Patch(facecolor='none', edgecolor='#8f1402', linewidth=2, label='Fire Boundaries'),
]
ax.legend(handles=legend_elements, loc='upper right')

ax.set_xticks([])
ax.set_yticks([])

plt.tight_layout()
plt.show()